# Start the CEDA MCP server

Runs the server inside this notebook's event loop (no subprocess). Open the notebook from the repository root (the folder containing `mcp_server.py`) and run the cells in order.

## 1. Install dependencies into this kernel

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import sys

import netCDF4
import xarray

print(sys.executable)
print("netCDF4", netCDF4.__version__, "| xarray", xarray.__version__)

## 2. Start the server as a background task

In [ ]:
import asyncio
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / "mcp_server.py").exists(), "Open the notebook from the repository root"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mcp_server import mcp

PORT = 8000
URL = f"http://localhost:{PORT}/mcp"

server_task = asyncio.create_task(
    mcp.run_async(transport="streamable-http", port=PORT, show_banner=False)
)
await asyncio.sleep(3)
if server_task.done():
    print("Server exited:", server_task.exception())
else:
    print(f"Server running at {URL}")

## 3. Check it: list tools and call `ceda_run`

In [ ]:
from fastmcp import Client


async def check():
    async with Client(URL, timeout=600) as client:
        print([t.name for t in await client.list_tools()])
        return await client.call_tool(
            "ceda_run",
            {
                "ecv": "SOILMOISTURE",
                "start_date": "2020-01-01T00:00:00Z",
                "end_date": "2020-01-05T23:59:59Z",
                "place": "Rome",
                "max_files": 3,
            },
        )


result = (await check()).data
print(result["bbox"], result["truncated"], len(result["records"]))
for d in result["datasets"]:
    print(d["uuid"], d["title"], d["inspect"]["variables"])

## 4. Stop the server

Skip this cell while you want other clients (e.g. your agent) to use `http://localhost:8000/mcp`. Restarting the kernel also stops it.

In [ ]:
server_task.cancel()
try:
    await server_task
except asyncio.CancelledError:
    pass
print("Server stopped")